# DDRI Station-Hour Model Comparison

대표 대여소 15개 `station-hour` 데이터셋을 사용해 추천 모델을 비교한다.

- Train: `2023`
- Validation: `2024`
- Final Train: `2023 + 2024`
- Test: `2025`
- 비교 모델: `LightGBM`, `CatBoost`
- objective 비교: `RMSE`, `Poisson`


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')
RAW_DATA_DIR = ROOT / 'works/05_prediction_long/data'
OUTPUT_DATA_DIR = ROOT / 'works/05_prediction_long/output/data'
IMAGE_DIR = ROOT / 'works/05_prediction_long/output/images'
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DATA_DIR / 'ddri_prediction_long_train_2023_2024.csv'
TEST_PATH = RAW_DATA_DIR / 'ddri_prediction_long_test_2025.csv'


## 1. 데이터 로드

대표 대여소 15개 `station-hour` 데이터셋을 읽고 `datetime`을 재구성한다.

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

for df in [train, test]:
    df['date'] = pd.to_datetime(df['date'])
    df['datetime'] = pd.to_datetime(
        df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['hour'].astype(str).str.zfill(2) + ':00:00'
    )

train.shape, test.shape

((263160, 16), (131400, 16))

## 2. 시계열 피처 생성

시간 단위 수요는 직전 1시간, 24시간, 1주 전 패턴의 영향을 많이 받을 수 있으므로 `lag`와 `rolling` 피처를 추가한다.

In [4]:
combined = pd.concat([train.assign(dataset='train'), test.assign(dataset='test')], ignore_index=True)
combined = combined.sort_values(['station_id', 'datetime']).reset_index(drop=True)

grouped_target = combined.groupby('station_id')['rental_count']
combined['lag_1h'] = grouped_target.shift(1)
combined['lag_24h'] = grouped_target.shift(24)
combined['lag_168h'] = grouped_target.shift(168)
combined['rolling_mean_24h'] = grouped_target.shift(1).rolling(24).mean()
combined['rolling_mean_168h'] = grouped_target.shift(1).rolling(168).mean()
combined['rolling_std_24h'] = grouped_target.shift(1).rolling(24).std()

train_model = combined[combined['dataset'] == 'train'].copy()
test_model = combined[combined['dataset'] == 'test'].copy()

FEATURE_COLUMNS = [
    'station_id', 'station_group', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_mean_168h', 'rolling_std_24h'
]
CATEGORICAL_COLUMNS = ['station_id', 'station_group', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']

train_model[FEATURE_COLUMNS].head(3)

,station_id,station_group,cluster,mapped_dong_code,hour,weekday,month,holiday,temperature,humidity,precipitation,wind_speed,lag_1h,lag_24h,lag_168h,rolling_mean_24h,rolling_mean_168h,rolling_std_24h
0,2312,주거 도착형,2,11680565,0,6,1,1,-2.0,81.0,0.0,9.3,NaN,NaN,NaN,NaN,NaN,NaN
1,2312,주거 도착형,2,11680565,1,6,1,1,-1.7,82.0,0.0,9.3,0.0,NaN,NaN,NaN,NaN,NaN
2,2312,주거 도착형,2,11680565,2,6,1,1,-0.8,77.0,0.0,10.8,0.0,NaN,NaN,NaN,NaN,NaN


## 3. 연 단위 분할

계절성과 날씨 분포를 반영하기 위해 `2023` 학습, `2024` validation, `2025` 테스트 구조를 사용한다.

In [5]:
train_2023 = train_model[train_model['date'] < '2024-01-01'].copy()
valid_2024 = train_model[train_model['date'] >= '2024-01-01'].copy()
full_train = train_model.copy()

X_train = train_2023[FEATURE_COLUMNS].copy()
y_train = train_2023['rental_count'].copy()

X_valid = valid_2024[FEATURE_COLUMNS].copy()
y_valid = valid_2024['rental_count'].copy()

X_full = full_train[FEATURE_COLUMNS].copy()
y_full = full_train['rental_count'].copy()

X_test = test_model[FEATURE_COLUMNS].copy()
y_test = test_model['rental_count'].copy()

for frame in [X_train, X_valid, X_full, X_test]:
    for column in CATEGORICAL_COLUMNS:
        frame[column] = frame[column].astype('category')

train_2023.shape, valid_2024.shape, full_train.shape, test_model.shape

((131400, 23), (131760, 23), (263160, 23), (131400, 23))

## 4. 모델 비교

추천 모델 문서 기준으로 `LightGBM`과 `CatBoost`를 우선 비교한다.
또한 카운트 데이터 특성을 반영하기 위해 `RMSE`와 `Poisson` objective를 함께 본다.

In [6]:
def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
    }


results = []

lightgbm_rmse = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='regression',
)
lightgbm_rmse.fit(X_train, y_train, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_RMSE', 'validation_2024', y_valid, lightgbm_rmse.predict(X_valid)))
lightgbm_rmse.fit(X_full, y_full, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_RMSE', 'test_2025_refit', y_test, lightgbm_rmse.predict(X_test)))

lightgbm_poisson = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='poisson',
)
lightgbm_poisson.fit(X_train, y_train, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_Poisson', 'validation_2024', y_valid, lightgbm_poisson.predict(X_valid)))
lightgbm_poisson.fit(X_full, y_full, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_Poisson', 'test_2025_refit', y_test, lightgbm_poisson.predict(X_test)))

catboost_rmse = CatBoostRegressor(
    loss_function='RMSE',
    iterations=300,
    depth=8,
    learning_rate=0.05,
    random_seed=42,
    verbose=False,
)
catboost_rmse.fit(X_train, y_train, cat_features=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('CatBoost_RMSE', 'validation_2024', y_valid, catboost_rmse.predict(X_valid)))
catboost_rmse.fit(X_full, y_full, cat_features=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('CatBoost_RMSE', 'test_2025_refit', y_test, catboost_rmse.predict(X_test)))

catboost_poisson = CatBoostRegressor(
    loss_function='Poisson',
    iterations=300,
    depth=8,
    learning_rate=0.05,
    random_seed=42,
    verbose=False,
)
catboost_poisson.fit(X_train, y_train, cat_features=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('CatBoost_Poisson', 'validation_2024', y_valid, catboost_poisson.predict(X_valid)))
catboost_poisson.fit(X_full, y_full, cat_features=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('CatBoost_Poisson', 'test_2025_refit', y_test, catboost_poisson.predict(X_test)))

results_df = pd.DataFrame(results)
results_df

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002721 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1490
[LightGBM] [Info] Number of data points in the train set: 131400, number of used features: 18
[LightGBM] [Info] Start training from score 0.722405


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017913 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1527
[LightGBM] [Info] Number of data points in the train set: 263160, number of used features: 18
[LightGBM] [Info] Start training from score 0.724514


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002259 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1490
[LightGBM] [Info] Number of data points in the train set: 131400, number of used features: 18
[LightGBM] [Info] Start training from score -0.325170


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003889 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1527
[LightGBM] [Info] Number of data points in the train set: 263160, number of used features: 18
[LightGBM] [Info] Start training from score -0.322255


,model,split,rmse,mae,r2
0,LightGBM_RMSE,validation_2024,1.0066,0.6121,0.5703
1,LightGBM_RMSE,test_2025_refit,0.8927,0.5455,0.5608
2,LightGBM_Poisson,validation_2024,1.0003,0.6074,0.5757
3,LightGBM_Poisson,test_2025_refit,0.8967,0.5402,0.5568
4,CatBoost_RMSE,validation_2024,1.0088,0.6139,0.5685
5,CatBoost_RMSE,test_2025_refit,0.9007,0.5488,0.5528
6,CatBoost_Poisson,validation_2024,1.0081,0.6095,0.5691
7,CatBoost_Poisson,test_2025_refit,0.9049,0.5460,0.5487


## 5. 결과 저장

비교표와 현재 우세 모델인 `LightGBM_RMSE`의 feature importance를 저장한다.

In [7]:
metrics_path = OUTPUT_DATA_DIR / 'ddri_station_hour_model_metrics.csv'
results_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')

importance_df = pd.DataFrame(
    {
        'feature': FEATURE_COLUMNS,
        'importance': lightgbm_rmse.feature_importances_,
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

importance_path = OUTPUT_DATA_DIR / 'ddri_station_hour_lightgbm_feature_importance.csv'
importance_df.to_csv(importance_path, index=False, encoding='utf-8-sig')

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(12), x='importance', y='feature', hue='feature', legend=False, palette='Blues_r')
plt.title('DDRI Station-Hour LightGBM Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
feature_image_path = IMAGE_DIR / 'ddri_station_hour_lightgbm_feature_importance.png'
plt.savefig(feature_image_path, dpi=150)
plt.close()

metrics_path, importance_path, feature_image_path

(PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/data/ddri_station_hour_model_metrics.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/data/ddri_station_hour_lightgbm_feature_importance.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/images/ddri_station_hour_lightgbm_feature_importance.png'))

## 6. 해석 메모

- 현재 실험에서는 `LightGBM_RMSE`가 2025 테스트 기준 가장 안정적인 성능을 보이면 기본 모델 후보로 채택한다.
- `Poisson` objective는 validation에선 매력적일 수 있지만, 최종 test에서 항상 우세한지 별도로 확인해야 한다.
- `CatBoost`는 범주형 처리 면에서 강점이 있지만, 이번 대표 대여소 15개 데이터에선 실제 성능으로 다시 판단한다.
